# Climate-Friendly Food Systems (CFFS) Labelling Project

### The University of British Columbia

****

## Baseline Calculation

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import xml.etree.ElementTree as et
from xml.etree.ElementTree import parse
from datetime import datetime
from configparser import ConfigParser

In [2]:
# RUN ONLY ONCE
# os.chdir is used to change the current directory to the specified path
os.chdir("../") # Sets path to the repo folder as it is one level above where this file exists!
path = os.getcwd()
print(path)

/Users/ankurbhardwaj/Desktop/SEEDS/CFFS_Label_2024_25


In [3]:
# menu_list = []
# # FOR OK
# menu_list.append(pd.read_csv('data/Misc/data_for_calculating_baseline/OK_2019.csv'))

# # FOR GATHER
# menu_list.append(pd.read_csv('data/Misc/data_for_calculating_baseline/Gather_2019.csv'))

# # FOR TOTEM
# menu_list.append(pd.read_csv('data/Misc/data_for_calculating_baseline/Totem_2019.csv'))

# recipes = pd.concat(menu_list)
# recipes

recipes = pd.read_csv('new_baseline_data.csv')

In [4]:
# recipes = recipes.drop(columns=["Unnamed: 13"])

In [5]:
recipes = recipes[recipes['Weight (g)'] >= 6]

In [6]:
recipes

,ProdId,Description,SalesGroup,GHG Emission (g),Weight (g),GHG Emission (kg),GHG Emission (kg) / 100g,GHG Emission (g) / 100g,Sales,GHG Emission (g) / 100g*Sales,N lost (g) / 100g,Eutrophying (g) / 100g,Freshwater Withdrawals (L) / 100g,Stress-Weighted Water Use (L) / 100g,Land Use (m^2) / 100g
0,R-54282,BLOWOUT|PomPom,FEAST,126.500000,275.0,0.126500,0.04600,46.00,117.0,5382.00,0.50,0.35,5.91,275.42,0.09
1,R-54281,BLOWOUT|YamChikPatty w CZR,FEAST,728.052686,335.5,0.728053,0.21701,217.01,14.0,3038.14,0.78,0.85,46.24,1964.84,0.31
2,R-44315,BNO|Bowl|Ancho Chicken,FT BUENO,1731.360106,505.0,1.731360,0.34284,342.84,4062.0,1392616.08,3.01,1.83,50.85,1231.93,0.42
3,R-44308,BNO|Bowl|Pulled Pork,FT BUENO,2051.300468,455.0,2.051300,0.45084,450.84,1699.0,765977.16,3.88,2.81,86.50,2822.19,0.69
4,R-44310,BNO|Bowl|Steak,FT BUENO,11452.096824,400.0,11.452097,2.86302,2863.02,73.0,209000.46,10.77,11.03,167.53,5340.72,8.81
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683,R-56915,HS|Rosemary Potatoes|SIDE,FT SIDES,83.536316,150.0,0.083536,0.05569,55.69,23.0,1280.87,0.50,0.40,6.55,303.87,0.11
684,R-44550,HS|Side Herbed Mashed Potatoes,FT HOMESKILLET,277.247059,180.0,0.277247,0.15403,154.03,70.0,10782.10,1.04,0.73,31.01,1365.94,0.43
685,R-30537,HS|Smokie|SIde,FT HOMESKILLET,1378.720000,112.0,1.378720,1.23100,1231.00,282.0,347142.00,13.28,7.64,179.58,6686.74,1.74
686,R-51298,HS|White Bean Cassoulet,FT HOMESKILLET,268.524958,351.0,0.268525,0.07650,76.50,62.0,4743.00,0.40,0.56,12.84,631.15,0.41


In [7]:
recipes["N lost (g) / 100g*Sales"] = recipes["N lost (g) / 100g"] * recipes["Sales"]
recipes["Land use (m^2) / 100g*Sales"] = recipes["Land Use (m^2) / 100g"] * recipes["Sales"]
recipes["Stress-Weighted Water Use (L) / 100g*Sales"] = recipes["Stress-Weighted Water Use (L) / 100g"] * recipes["Sales"]
recipes["Freshwater Withdrawals (L) / 100g*Sales"] = recipes["Freshwater Withdrawals (L) / 100g"] * recipes["Sales"]
recipes["Eutrophying (g) / 100g*Sales"] = recipes["Eutrophying (g) / 100g"] * recipes["Sales"]
recipes

,ProdId,Description,SalesGroup,GHG Emission (g),Weight (g),GHG Emission (kg),GHG Emission (kg) / 100g,GHG Emission (g) / 100g,Sales,GHG Emission (g) / 100g*Sales,N lost (g) / 100g,Eutrophying (g) / 100g,Freshwater Withdrawals (L) / 100g,Stress-Weighted Water Use (L) / 100g,Land Use (m^2) / 100g,N lost (g) / 100g*Sales,Land use (m^2) / 100g*Sales,Stress-Weighted Water Use (L) / 100g*Sales,Freshwater Withdrawals (L) / 100g*Sales,Eutrophying (g) / 100g*Sales
0,R-54282,BLOWOUT|PomPom,FEAST,126.500000,275.0,0.126500,0.04600,46.00,117.0,5382.00,0.50,0.35,5.91,275.42,0.09,58.50,10.53,32224.14,691.47,40.95
1,R-54281,BLOWOUT|YamChikPatty w CZR,FEAST,728.052686,335.5,0.728053,0.21701,217.01,14.0,3038.14,0.78,0.85,46.24,1964.84,0.31,10.92,4.34,27507.76,647.36,11.90
2,R-44315,BNO|Bowl|Ancho Chicken,FT BUENO,1731.360106,505.0,1.731360,0.34284,342.84,4062.0,1392616.08,3.01,1.83,50.85,1231.93,0.42,12226.62,1706.04,5004099.66,206552.70,7433.46
3,R-44308,BNO|Bowl|Pulled Pork,FT BUENO,2051.300468,455.0,2.051300,0.45084,450.84,1699.0,765977.16,3.88,2.81,86.50,2822.19,0.69,6592.12,1172.31,4794900.81,146963.50,4774.19
4,R-44310,BNO|Bowl|Steak,FT BUENO,11452.096824,400.0,11.452097,2.86302,2863.02,73.0,209000.46,10.77,11.03,167.53,5340.72,8.81,786.21,643.13,389872.56,12229.69,805.19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683,R-56915,HS|Rosemary Potatoes|SIDE,FT SIDES,83.536316,150.0,0.083536,0.05569,55.69,23.0,1280.87,0.50,0.40,6.55,303.87,0.11,11.50,2.53,6989.01,150.65,9.20
684,R-44550,HS|Side Herbed Mashed Potatoes,FT HOMESKILLET,277.247059,180.0,0.277247,0.15403,154.03,70.0,10782.10,1.04,0.73,31.01,1365.94,0.43,72.80,30.10,95615.80,2170.70,51.10
685,R-30537,HS|Smokie|SIde,FT HOMESKILLET,1378.720000,112.0,1.378720,1.23100,1231.00,282.0,347142.00,13.28,7.64,179.58,6686.74,1.74,3744.96,490.68,1885660.68,50641.56,2154.48
686,R-51298,HS|White Bean Cassoulet,FT HOMESKILLET,268.524958,351.0,0.268525,0.07650,76.50,62.0,4743.00,0.40,0.56,12.84,631.15,0.41,24.80,25.42,39131.30,796.08,34.72


## Baseline Calculation

The baseline calculation is done according to the followinf formula

$$emission_Baseline = \frac{\sum (Emission_{restaurant} * Sales_{restaurant})}{\sum Sales_{restaurant}}$$

where the restaurants are Open Kitchen, Gather and Feast.

In [8]:
GHG_Baseline = sum(recipes["GHG Emission (g) / 100g*Sales"]) / sum(recipes["Sales"])
land_Baseline = sum(recipes["Land use (m^2) / 100g*Sales"]) / sum(recipes["Sales"])
nitrogen_Baseline = sum(recipes["N lost (g) / 100g*Sales"]) / sum(recipes["Sales"])
stress_water_Baseline = sum(recipes["Stress-Weighted Water Use (L) / 100g*Sales"]) / sum(recipes["Sales"])
fresh_water_Baseline = sum(recipes["Freshwater Withdrawals (L) / 100g*Sales"]) / sum(recipes["Sales"])
eutro_Baseline = sum(recipes["Eutrophying (g) / 100g*Sales"]) / sum(recipes["Sales"])
GHG_Baseline,land_Baseline,nitrogen_Baseline,stress_water_Baseline, fresh_water_Baseline, eutro_Baseline

(691.590165684274,
 1.6073181747336205,
 4.672197064157782,
 2818.650105937804,
 83.75048712262895,
 3.257670182970605)

In [9]:
config = ConfigParser()
config["baseline"] = {"GHG_Baseline" : GHG_Baseline,
                      "land_baseline": land_Baseline,
                      "nitrogen_Baseline": nitrogen_Baseline,
                      "stress_water_Baseline": stress_water_Baseline,
                      "fresh_water_Baseline": fresh_water_Baseline,
                      "eutro_Baseline": eutro_Baseline}
with open("data/Misc/data_for_calculating_baseline/baseline3.ini", "w") as f:
    config.write(f)